In [4]:
import os
import csv
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [5]:
from pathlib import Path

def resolve_repo_root():
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent.parent)
    candidates.append(Path.cwd())
    for candidate in candidates:
        if (candidate / "data" / "processed" / "train.csv").exists():
            return candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "processed" / "train.csv").exists():
            return candidate
    return candidates[0] if candidates else Path.cwd()

REPO_ROOT = resolve_repo_root()
TRAIN_CSV = REPO_ROOT / "data" / "processed" / "train.csv"
VAL_CSV = REPO_ROOT / "data" / "processed" / "val.csv"

FEATURE_COLS = [
    "num_vars", "num_assertions", "num_uninterpreted_funcs",
    "num_func_applications", "max_func_nesting_depth",
    "ast_node_count", "max_depth", "num_arith_ops", "file_size_bytes"
]
TARGET_COL = "result"

In [6]:
def load_split(path):
    df = pd.read_csv(path)
    X = df[FEATURE_COLS]
    y = df[TARGET_COL]
    return X, y

In [7]:
X_train, y_train = load_split(TRAIN_CSV)
X_val, y_val = load_split(VAL_CSV)

# Baseline: always predict the majority class
majority_class = y_train.value_counts().idxmax()
baseline_preds = [majority_class] * len(y_val)
baseline_acc = accuracy_score(y_val, baseline_preds)
print(f"Majority-class baseline ({majority_class}): {baseline_acc:.3f} accuracy\n")

# Decision tree, depth-limited so it stays readable
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

val_preds = clf.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Decision tree validation accuracy: {val_acc:.3f}\n")

print("Classification report:")
print(classification_report(y_val, val_preds))

print("Confusion matrix (rows=true, cols=predicted):")
labels = sorted(y_train.unique())
print("Labels order:", labels)
print(confusion_matrix(y_val, val_preds, labels=labels))

print("\nLearned tree structure:")
print(export_text(clf, feature_names=FEATURE_COLS))

print("\nFeature importances:")
for name, importance in sorted(zip(FEATURE_COLS, clf.feature_importances_), key=lambda x: -x[1]):
    print(f"  {name}: {importance:.3f}")

Majority-class baseline (sat): 0.641 accuracy

Decision tree validation accuracy: 0.891

Classification report:
              precision    recall  f1-score   support

         sat       0.91      0.95      0.93        41
     unknown       0.83      1.00      0.91         5
       unsat       0.87      0.72      0.79        18

    accuracy                           0.89        64
   macro avg       0.87      0.89      0.88        64
weighted avg       0.89      0.89      0.89        64

Confusion matrix (rows=true, cols=predicted):
Labels order: ['sat', 'unknown', 'unsat']
[[39  0  2]
 [ 0  5  0]
 [ 4  1 13]]

Learned tree structure:
|--- file_size_bytes <= 5980.50
|   |--- num_func_applications <= 161.50
|   |   |--- max_func_nesting_depth <= 2.00
|   |   |   |--- ast_node_count <= 21.50
|   |   |   |   |--- max_func_nesting_depth <= 0.50
|   |   |   |   |   |--- class: unsat
|   |   |   |   |--- max_func_nesting_depth >  0.50
|   |   |   |   |   |--- class: sat
|   |   |   |--- ast_